# Powerflow Plotting Quick Test

This notebook is the Step 5 postprocessing workspace for DB-backed Step 4 power-flow results. It is meant to be a compact review notebook: first inspect one network/timestep spatially, then compare voltage, transformer loading, and line-loading stress summaries that scale from the current dummy grid to a larger grid population.

Use the Python kernel from `GridExpand/5.postprocessing/.venv` so the plotting and notebook dependencies are available.

Workflow:

- The setup cell resolves the repository root, imports the Step 5 plotting helpers, and selects one DB-backed power-flow run through `db_grid`, `run_name`, and `stage`.
- The network plot is for detailed spatial inspection of one timestep. Use it after the summary plots point to an interesting stage or timestep.
- The voltage and loading distribution plots are for population-level diagnosis. They are designed so the same notebook still works after the Munich batch contains many grids.

Current scope:

- `db_grid` selects the existing PLZ 94342 dummy test grid.
- `run_name = "baseline_static_full_powerflow"` selects the full Step 4 DB run.
- `stage = "post"` is used as the default detailed-inspection stage, while comparison plots explicitly load both `pre` and `post` where needed.


In [ ]:
from pathlib import Path
import sys

# Resolve repository root even if notebook opens with a different working directory.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
root = next((p for p in candidates if (p / "GridExpand").exists()), None)
if root is None:
    raise RuntimeError("Could not find repository root containing 'GridExpand'.")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

plotting_dir = root / "GridExpand/5.postprocessing/plotting"
if str(plotting_dir) not in sys.path:
    sys.path.insert(0, str(plotting_dir))

root


In [ ]:
from IPython.display import display
import importlib
import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go

import powerflow_plotting
importlib.reload(powerflow_plotting)

from powerflow_plotting import (
    db_powerflow_timestep_bounds,
    grid_loading_stress_summary,
    line_loading_distribution_db,
    plot_grid_loading_stress_scatter,
    plot_line_loading_ecdf,
    plot_powerflow_heatmap_db,
    plot_transformer_apparent_power_stage_comparison_matplotlib,
    plot_transformer_import_distributions_matplotlib,
    plot_voltage_deviation_histogram,
    transformer_import_distribution_db,
    voltage_deviation_summary_db,
)

# DB-backed Step 4 result run produced by the recent PLZ 94342 smoke test.
db_grid = "9278140-00_94342_1_-1.h5"
run_name = "baseline_static_full_powerflow"
stage = "post"

bounds = db_powerflow_timestep_bounds(db_grid, stage=stage, run_name=run_name)
default_timestep = bounds["min_timestep"]
bounds


## Network Plot

The slider redraws one DB-backed network heatmap in a single output area. It is intended for local inspection after the summary plots identify an interesting timestep, grid, or stress pattern.

How to read it:

- Bus color shows voltage magnitude in p.u. for the selected timestep.
- Line color shows loading relative to the line current rating.
- Hover labels expose the underlying bus voltage or line loading values.
- The slider changes `t_index` and redraws the same figure area, so the notebook should show only one active network plot.

What to look for:

- Spatial clusters of low or high voltage, especially near feeder ends.
- Lines with loading close to or above 100%, and whether they sit on the feeder backbone or only on short service branches.
- Whether voltage and line-loading issues occur in the same area, which can indicate a locally constrained feeder section.

Use this plot as a diagnostic lens, not as the primary population metric: once hundreds of grids are available, first use the stress summaries below to select which grid/timestep deserves detailed spatial inspection.


In [ ]:
timestep_slider = widgets.IntSlider(
    value=default_timestep,
    min=bounds["min_timestep"],
    max=bounds["max_timestep"],
    step=1,
    description="Timestep",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
)


def build_network_figure(timestep):
    return plot_powerflow_heatmap_db(
        input_id=db_grid,
        stage=stage,
        timestep=timestep,
        on_map=False,
        map_style="light",
        cmap="Jet",
        climits_volt=(0.9, 1.1),
        climits_load=(0.0, 100.0),
        show_household_buses=False,
        show=False,
        run_name=run_name,
    )


initial_network_figure = build_network_figure(timestep_slider.value)
network_figure = go.FigureWidget(initial_network_figure)


def render_timestep(change=None):
    fig = build_network_figure(timestep_slider.value)
    with network_figure.batch_update():
        network_figure.data = []
        network_figure.add_traces(fig.data)
        network_figure.layout = fig.layout


timestep_slider.observe(render_timestep, names="value")
display(widgets.VBox([timestep_slider, network_figure]))


## Voltage Deviation Histogram

This mirrors the attached voltage-limit figure by comparing each grid's minimum and maximum voltage magnitude against the 0.9 p.u. and 1.1 p.u. limits.

How to read it:

- The blue histogram summarizes each grid's maximum upper-voltage value across all buses and timesteps in the selected stage.
- The green histogram summarizes each grid's minimum lower-voltage value across all buses and timesteps in the selected stage.
- The dashed vertical lines mark the 0.9 p.u. and 1.1 p.u. voltage thresholds.
- The annotations report the share of grids outside the voltage limits.

What to look for:

- A right-shifted blue tail beyond 1.1 p.u. indicates overvoltage risk, often relevant for feed-in-heavy scenarios.
- A left-shifted green tail below 0.9 p.u. indicates undervoltage risk, often relevant for high-load electrification scenarios.
- A narrow distribution close to 1.0 p.u. means most grids remain voltage-stable under the selected scenario.
- When the Munich batch is available, compare this plot between `pre` and `post` stages or between scenario runs to see whether electrification shifts the voltage-risk tail.


In [ ]:
voltage_summary = voltage_deviation_summary_db(
    db_grid,
    run_name=run_name,
    stages=(stage,),
)

display(
    voltage_summary[[
        "grid",
        "stage",
        "n_timesteps",
        "n_buses",
        "min_vm_pu",
        "max_vm_pu",
    ]].style.format({
        "min_vm_pu": "{:.4f}",
        "max_vm_pu": "{:.4f}",
    })
)

display(plot_voltage_deviation_histogram(voltage_summary, show=False))


## Transformer Import Distributions

This section uses the thesis plotting style from `thesis_plots.ipynb`: daily aggregation for the time-series panels, hourly load-duration curves, Gaussian-equivalent 68% and 96% percentile bands, and LDC normalization by the average per-grid maximum apparent load.

How to read it:

- Rows correspond to active power `P`, reactive power `|Q|`, and apparent power `|S|` transformer import.
- The left column shows daily aggregated seasonal behavior over the year.
- The right column shows load-duration curves, so high values near 0% duration are rare peaks and values toward 100% duration are persistent base levels.
- The dark line is the expected profile across grids. The darker and lighter bands show 68% and 96% percentile ranges.

What to look for:

- Active power `P` shows the dominant net import seasonality and peak-load behavior.
- Reactive power `|Q|` helps check whether reactive assumptions or compensation behavior materially affect transformer loading.
- Apparent power `|S|` is the transformer-loading quantity to watch for thermal capacity stress.
- Wide percentile bands mean grid-to-grid diversity is high; narrow bands mean the selected population behaves similarly.
- In the LDC panels, a steep left edge means short peak events dominate. A high curve over a broad duration range means sustained loading.

Use this full 3x2 plot when you want to understand the composition of transformer import. Use the next apparent-power-only pre/post plot when the question is specifically how electrification changes transformer loading shape.


In [ ]:
transformer_distribution = transformer_import_distribution_db(
    db_grid,
    run_name=run_name,
    stage=stage,
)

transformer_fig = plot_transformer_import_distributions_matplotlib(
    transformer_distribution,
    show=True,
)


## Apparent Transformer Load: Pre vs Post

This compares the daily aggregated apparent transformer load for the pre/status-quo power-flow stage on the left and the post/electrification stage on the right.

How to read it:

- Each panel shows the normalized apparent transformer load `|S|` over the year after daily aggregation.
- The dark line is the expected daily profile across the selected grid population.
- The darker and lighter bands show the 68% and 96% percentile ranges across grids, so wider bands mean stronger grid-to-grid diversity.
- The left panel is the electricity-only/status-quo power-flow case. The right panel is the post/electrification case.

What to look for:

- A higher or broader winter peak in the post panel indicates heat-pump or electrification-driven transformer stress.
- A flatter post profile can indicate that the new loads are spread more evenly over the year.
- Because this plot is normalized per stage, it is best for comparing seasonal shape and variability, not absolute MVA growth. Use transformer ratings or absolute-load plots if the question is capacity sizing.


In [ ]:
transformer_pre_distribution = transformer_import_distribution_db(
    db_grid,
    run_name=run_name,
    stage="pre",
)
transformer_post_distribution = transformer_import_distribution_db(
    db_grid,
    run_name=run_name,
    stage="post",
)

apparent_power_stage_fig = plot_transformer_apparent_power_stage_comparison_matplotlib(
    transformer_pre_distribution,
    transformer_post_distribution,
    show=True,
)


## Line Loading Stress Across Grids

This section summarizes each line by its maximum loading over all timesteps, then compares the full line-loading distribution and robust per-grid stress metrics between the pre/status-quo and post/electrification stages.

For the current dummy run this is filtered to `db_grid`; for a larger Munich batch, call `line_loading_distribution_db(input_id=None, run_name=run_name, ags=<munich_ags>)` or use another DB filter.

How to read the table:

- `median_line_max_loading_percent` describes the typical line in a grid after taking each line's worst timestep.
- `p95_line_max_loading_percent` is the main robust stress indicator: it highlights the high-loading tail without depending on one single line.
- `max_line_loading_percent` is still useful for finding the worst case, but it should be interpreted together with p95 and overload shares.
- `share_lines_above_80_percent` shows how much of the grid is highly loaded. `share_lines_above_100_percent` shows how much is overloaded.

How to read the ECDF:

- The x-axis is each line's maximum loading over all timesteps.
- The y-axis is the share of line maxima below that loading.
- Curves shifted to the right are more stressed. A long right tail means only a small set of lines is critical; a broad shift means the whole grid population is more heavily loaded.
- The 80% and 100% vertical lines mark high-loading and overload reference levels.

How to read the pre-vs-post scatter:

- Each point is one grid. The x-axis is the pre-stage p95 line maximum loading, and the y-axis is the post-stage p95 line maximum loading.
- Points above the diagonal became more stressed after electrification. Points below the diagonal became less stressed.
- Marker color shows the post-stage share of overloaded lines, so dark points are priority candidates for reinforcement analysis.
- Marker size scales with the number of lines, which helps distinguish small grids from larger grids in the Munich batch.


In [ ]:
line_loading = line_loading_distribution_db(
    input_id=db_grid,
    run_name=run_name,
    stages=("pre", "post"),
)
line_stress = grid_loading_stress_summary(line_loading)

stress_table = line_stress[[
    "grid",
    "stage",
    "n_lines",
    "median_line_max_loading_percent",
    "p95_line_max_loading_percent",
    "max_line_loading_percent",
    "share_lines_above_80_percent",
    "share_lines_above_100_percent",
]].copy()

display(
    stress_table.style.format({
        "median_line_max_loading_percent": "{:.2f}%",
        "p95_line_max_loading_percent": "{:.2f}%",
        "max_line_loading_percent": "{:.2f}%",
        "share_lines_above_80_percent": "{:.1f}%",
        "share_lines_above_100_percent": "{:.1f}%",
    })
)

display(plot_line_loading_ecdf(line_loading, show=False))
display(plot_grid_loading_stress_scatter(line_stress, show=False))
